In [ ]:
import sys

if "google.colab" in sys.modules:
    import os

    repo = "/content/lmbike_workshop_gimbal"
    sim = "/content/lmbike_workshop_gimbal/bike_mujoco_sim"
    if not os.path.isdir(repo):
        !git clone https://github.com/innowing-robotics-interns/lmbike_workshop_gimbal.git /content/lmbike_workshop_gimbal
    %cd /content/lmbike_workshop_gimbal/bike_mujoco_sim
    if sim not in sys.path:
        sys.path.insert(0, sim)

    # Colab's Jupyter must stay the copy it shipped with. requirements.txt
    # does not list Jupyter. The constraint pins whatever Jupyter packages
    # are already installed so pip cannot replace them.
    import importlib.metadata as metadata
    protected = {
        "notebook", "jupyter", "jupyterlab", "jupyter-client", "jupyter-core",
        "jupyter-server", "ipykernel", "ipython", "nbformat", "nbclient",
        "nbconvert", "nbclassic",
    }
    pinned = []
    for dist in metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().replace("_", "-") in protected:
            pinned.append(f"{name}=={dist.version}")
    with open("/tmp/colab-jupyter-constraints.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(pinned) + "\n")
    !pip install -q --upgrade-strategy only-if-needed -r requirements.txt -c /tmp/colab-jupyter-constraints.txt


# Setup 0. Prepare your local notebook server

The local lessons need Python 3.10 or 3.11, the simulation packages, and a Jupyter kernel connected to that same environment. On macOS, the kernel must use MuJoCo's `mjpython` launcher so the window demonstrations can work.

Start with [HOW_TO_RUN_NOTEBOOK.md](../HOW_TO_RUN_NOTEBOOK.md) in a terminal. This notebook checks the environment after the local server opens. Google Colab users can keep using the first Colab cell; the local Jupyter installation below is not needed there.

- **Environment:** a venv or Conda environment containing the course's Python and packages.
- **Server:** the local JupyterLab program that your browser connects to.
- **Kernel:** the separate process that runs a notebook's cells. A correctly started server can still have the wrong kernel selected.


## Step A. Choose Python 3.10 or 3.11

Install Python 3.10 or 3.11 with your preferred installer, then create a venv or Conda environment as shown in [the run instructions](../HOW_TO_RUN_NOTEBOOK.md). Use a Python build that includes Tk for the mouse joystick. Conda users can install Tk with `conda install tk`; Homebrew users need the `python-tk` package matching their Python version.

Activate the environment before the next steps. You can keep other Python versions installed; what matters is selecting the correct environment for this course.


## Step B. Install the packages and prepare the kernel

In a terminal, activate your environment and go to `bike_mujoco_sim`, the folder containing `run_sim.py`. Then run:

```bash
python tutorials/setup_env.py
```

This checks the Python version, installs missing or incompatible packages from both requirements files, and writes this checkout's kernel definition. On macOS the definition starts `mjpython`; on Windows and Linux it starts ordinary Python. It uses your environment's paths, not the author's paths.

If you have already installed all the packages, they are kept. To only check packages and write the kernel definition, use `python tutorials/setup_env.py --skip-install`.


## Step C. Start the local server and choose its kernel

From that same environment and folder, run:

```bash
python start_notebooks.py
```

Open the local URL in the terminal if the browser does not open automatically. In JupyterLab open this notebook and select **Bike MuJoCo sim** as its kernel. The launcher exposes the project-local kernel to its server, so no per-user kernel registration is required.

If using VS Code, connect to this **existing Jupyter server URL** and select **Bike MuJoCo sim**. On macOS, choosing the raw Python environment instead skips the required mjpython launcher. Starting the server with mjpython alone does not change a generic Python kernel.


## Step D. Check what this notebook is using

Run the next cell. Python 3.10/3.11 and a venv or Conda environment are supported for local use. Every dependency should say `OK`. On macOS, the final dispatcher check should also say `True`.

If the macOS dispatcher check is False, select **Bike MuJoCo sim**, restart the kernel, and run from the top. If a package is missing, repair the environment with Step E or F, then restart this kernel. Colab uses its provided Python and does not need the local kernel checks.


In [ ]:
import os
import sys
from pathlib import Path

print("Python version :", sys.version)
print("Executable     :", sys.executable)
print("Prefix         :", sys.prefix)
in_colab = "google.colab" in sys.modules
if in_colab:
    print("Using Colab's provided Python; local environment checks are skipped.")
elif sys.version_info[:2] not in ((3, 10), (3, 11)):
    print("PROBLEM: select Python 3.10 or 3.11 for the local server.")
else:
    print("Version looks OK.")

in_env = sys.prefix != sys.base_prefix or (Path(sys.prefix) / "conda-meta").is_dir()
print("Looks like a venv or Conda environment:", bool(in_env))
needed = [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
          ("yaml", "PyYAML"), ("mujoco", "mujoco"), ("ipykernel", "ipykernel")]
missing = []
for module_name, pip_name in needed:
    try:
        mod = __import__(module_name)
        print(f"OK   {module_name:12} {getattr(mod, '__version__', '?')}")
    except Exception as exc:
        missing.append(pip_name)
        print(f"MISS {module_name:12} -> {pip_name!r}: {exc}")
if missing:
    print("Run python tutorials/setup_env.py in the course environment.")
else:
    print("All required packages import cleanly.")
if sys.platform == "darwin" and not missing:
    from mujoco import viewer
    ready = viewer._MJPYTHON is not None
    print("macOS mjpython dispatcher:", ready)
    if not ready:
        print("Select Bike MuJoCo sim, restart its kernel, then run from the top.")


## Step E. Manual package installation, if needed

In a terminal with the course environment activated, from `bike_mujoco_sim`:

```bash
python -m pip install --no-user -r requirements.txt -r requirements-jupyter.txt
python tutorials/setup_env.py --skip-install
python start_notebooks.py
```

`import yaml` comes from the package **PyYAML**. Use `mujoco`, not `mujoco-py`.

These are local-server commands. Do not install `requirements-jupyter.txt` on Colab: Colab already supplies Jupyter, and replacing it can prevent the notebook from opening. The first code cell of each lesson installs simulation requirements only on Colab.


## Step F. Repair setup from this notebook

The next cell runs `tutorials/setup_env.py` with this kernel's own Python. Use it only after selecting the right environment. It checks existing packages, installs missing ones, and prepares this checkout's kernel definition. Restart the local server with `python start_notebooks.py` if you change environments, then choose **Bike MuJoCo sim** again.

On Colab the cell deliberately skips local setup.


In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Colab already supplies Jupyter; skipping local setup.")
else:
    import os
    import subprocess
    import sys

    def find_project_root(start=None):
        current = os.path.abspath(start or os.getcwd())
        while True:
            if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
                os.path.join(current, "requirements.txt")
            ):
                return current
            parent = os.path.dirname(current)
            if parent == current:
                break
            current = parent
        raise RuntimeError("Could not find bike_mujoco_sim. Open the folder that contains run_sim.py.")

    root = find_project_root()
    script = os.path.join(root, "tutorials", "setup_env.py")
    print("Using:", sys.executable)
    print("Running:", script)
    result = subprocess.run([sys.executable, script], cwd=root)
    print("exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(
            "Setup failed. Read the error above, activate the correct Python 3.10/3.11 "
            "venv or Conda environment, and run python tutorials/setup_env.py in a terminal."
        )


## Step G. Start Lesson 1

Run the environment check once more. Select **Bike MuJoCo sim** on the local server, then open `01_environment_and_the_fall.ipynb` and run from the top.

The 3D window demonstrations wait for you to close the MuJoCo window. In Lesson 4, the separate mouse joystick closes when the main MuJoCo window closes. The full local workflow and troubleshooting steps are in [HOW_TO_RUN_NOTEBOOK.md](../HOW_TO_RUN_NOTEBOOK.md).
